#### PII middleware will:

* Redact emails before sending input to LLM
* Mask credit cards before sending input to LLM

Example:

```text id="f1mlm8"
john@gmail.com
→ [REDACTED_EMAIL]

5105-1051-0510-5100
→ ****-****-****-5100
```

LangChain officially supports:

* `email`
* `credit_card`
* `ip`
* `mac_address`
* `url`

with strategies:

* `redact`
* `mask`
* `hash`
* `block`

---

#### Important Difference

| Strategy | Output                         |
| -------- | ------------------------------ |
| redact   | Completely removes value       |
| mask     | Shows partial value            |
| block    | Raises exception               |
| hash     | Converts to deterministic hash |


---

[LangChain Middleware Docs](https://docs.langchain.com/oss/python/langchain/middleware/built-in)


In [ ]:
from dotenv import load_dotenv
load_dotenv()

# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# model = init_chat_model("groq:qwen/qwen3-32b")
# model = init_chat_model("groq:llama-3.3-70b-versatile")
# PIIMiddleware("ip", strategy="hash"),

True

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[],
    middleware=[

        # Replace email with [REDACTED_EMAIL]
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        # Mask credit card except last 4 digits
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        
    ],
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "My email is john@gmail.com "
                "and card is 5105-1051-0510-5100 "
                "is mosquito a prasite"
                "my ip is 12.40.70.90 "
            )
        }
    ]
})


In [10]:
response['messages'][0].content

'My email is [REDACTED_EMAIL] and card is ****-****-****-5100 is mosquito a prasitemy ip is 12.40.70.90 '

| Feature           | Guardrails                                                          | PII Middleware                            |
| ----------------- | ------------------------------------------------------------------- | ----------------------------------------- |
| Main Purpose      | Overall safety, policy and behavior control                         | Detect and handle sensitive personal data |
| Scope             | Broad                                                               | Narrow/specific                           |
| Focus Area        | Safety, compliance, moderation, prompt injection, output validation | Emails, credit cards, IPs, API keys etc.  |
| Works On          | Input, output, tools, agent behavior                                | Mainly sensitive data in input/output     |
| Detection Type    | Deterministic + LLM-based                                           | Mostly deterministic                      |
| Uses LLM?         | Sometimes yes                                                       | Usually no                                |
| Speed             | Slower if model-based                                               | Very fast                                 |
| Cost              | Can increase due to extra model calls                               | Very cheap                                |
| Predictability    | May vary with model                                                 | Highly predictable                        |
| Typical Checks    | Toxicity, jailbreaks, policy violations, hallucinations             | Regex/Luhn-based PII detection            |
| Example           | Block prompt injection                                              | Mask credit card                          |
| Example Action    | Stop unsafe response                                                | Redact email                              |
| Enterprise Usage  | AI safety layer                                                     | Compliance/privacy layer                  |
| Common Strategies | Block, retry, validate, escalate                                    | Redact, mask, hash, block                 |
| Placement         | Around entire agent lifecycle                                       | Before/after model calls                  |
| LangChain Example | HumanInLoop, Moderation, Prompt validation                          | `PIIMiddleware("email")`                  |
| Best For          | Agent governance and AI safety                                      | GDPR/HIPAA/PCI compliance                 |
